# Part 2 — Comparing Multiple Protein Chains

In **Part 1** you learned how to load a protein structure, compute its Backbone
Rigid Invariants (BRI), and visualise a single chain with BID / BIB diagrams.

This notebook picks up where Part 1 left off and focuses on **comparison**:

- How do you compare the backbone geometry of **two or more chains**?
- Can you **quantify** how different they are?
- Which parts of the backbone are **rigid** and which are **flexible**?

## What you will learn

| Section | Method | Question answered |
|---------|--------|-------------------|
| 3 | **Overlay plots** | Where along the sequence do two chains differ? |
| 4 | **Distance matrix + heatmap** | How different are 20 chains, quantitatively? |
| 5 | **Scatter projection (BRI + LAI)** | How do whole chains differ in their global structural signatures and internal heterogeneity? |
| 6 | **Batch workflow** | How to process a whole folder of PDB files at once? |

## Example proteins

| Protein | PDB ID | Role in this notebook |
|---------|--------|-----------------------|
| Human hemoglobin | **1HHO** | Two chain types (α and β) for overlay comparison |
| NMR ensemble | **2K4P** | 20 models of the same protein for distance-matrix comparison |

> **Prerequisite:** Run Part 1 first if you are unfamiliar with BRI / LAI or
> how to load structures with `ProteinChain` / `ProteinEntry`.

## 1. Setup

In [ ]:
import bri
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from bri import ProteinEntry
from bri.invariant import BRI_COLUMNS, LAI_COLUMNS
from bri.invariant_compare import group_invariant_compare
from bri.workflow import distance_table_to_matrix, plot_invariant_curves

plt.rcParams["figure.dpi"] = 100
print(f"bri version: {bri.__version__}")

## 2. Preparing data for comparison

We will use two datasets:

1. **Hemoglobin** (1HHO) — to compare its α and β chains visually.
2. **An NMR ensemble** (2K4P) — 20 structural models of the same protein,
   all with the same chain length. This lets us compute pairwise distances
   between every pair of models.

We compute BRI for each chain and store the results. Recall from Part 1 that
`get_invariant(invariant_type="bri")` returns the nine coordinate-based
invariant values (`x(N)` … `z(C)`) plus metadata.

In [ ]:
# --- Hemoglobin: two chain types (different lengths) ---
hemo = ProteinEntry.from_cif("1hho")
chain_a = hemo["A"]   # α-chain 
chain_b = hemo["B"]   # β-chain 

bri_a = chain_a.get_invariant(invariant_type="bri")
bri_b = chain_b.get_invariant(invariant_type="bri")

In [ ]:
## Exercise: compare lengths of chain A and chain B.

In [ ]:
# --- NMR ensemble: 20 models (same length) ---
nmr_entry = ProteinEntry.from_cif("2k4p")
poly_chains = [c for c in nmr_entry.chains if c.polypeptide]

# Compute BRI for every model and combine into one table.
# Each model gets a unique label so they can be distinguished later.
frames = []
for chain in poly_chains:
    inv = chain.get_invariant(invariant_type="bri")
    inv["pdb_id"] = f"model_{chain.model_id}"
    frames.append(inv)

nmr_bri = pd.concat(frames, ignore_index=True)

n_models = nmr_bri["pdb_id"].nunique()
chain_len = int(nmr_bri["chain_length"].iloc[0])
print(f"NMR ensemble: {n_models} models × {chain_len} residues = {len(nmr_bri)} rows")

## 3. Overlay comparison — where do two chains differ?

The most direct way to compare two chains is to **plot their BRI values along
the residue sequence** and overlay them.

We use hemoglobin's α-chain (blue) and β-chain (orange). Although they share
the same overall fold (the *globin fold*), their local backbone geometry differs
at many positions — these differences are what the overlay reveals.

**How to read the plots:**

- Each panel is one coordinate of one backbone atom (N, Cα, or C).
- **Flat regions** → regular secondary structure (α-helix, β-strand).
- **Spikes or bends** → turns, loops, or unusual local geometry.
- **Where blue and orange diverge** → the two chains have different backbone
  geometry at that position.

In [ ]:
fig = plot_invariant_curves(frames, BRI_COLUMNS)
# fig.suptitle("BRI Overlay — Hemoglobin α (blue) vs β (orange)", fontsize=13, y=1.01)
fig.tight_layout()
plt.show()

## 4. Pairwise distance matrix

Overlay plots are great for two chains, but what about **20 or more**?
The function `group_invariant_compare()` computes the BRI distance between
every pair of chains in one call, then `distance_table_to_matrix()` converts
the result into a symmetric matrix for heat-map visualisation.

### Two distance metrics

| Metric | What it measures | Character |
|--------|-----------------|-----------|
| **Chebyshev** (L∞) | the maximum absolute difference between any of the corresponding coordinates of two points| Conservative — reports the worst mismatch |
| **RMS** (root-mean-square) | Square root of the mean of squared differences between the corresponding coordinates | "Average" overall comparison |


> **Requirement:** pairwise comparison requires all chains to have the **same
> length**.   
> To compare chains of different lengths, use the statistical approach in
> Section 5.


The NMR ensemble satisfies this (all 20 models have 65 residues).  

In [ ]:
# Compute pairwise distances with both metrics
dist_cheb = group_invariant_compare(nmr_bri, metric="chebyshev")
dist_rms = group_invariant_compare(nmr_bri, metric="rms")

mat_cheb = distance_table_to_matrix(dist_cheb)
mat_rms = distance_table_to_matrix(dist_rms)

print(f"Distance tables: {len(dist_cheb)} pairs each")
print(f"Distance matrix size: {mat_cheb.shape[0]} × {mat_cheb.shape[1]}")

### Heat-map

We plot both metrics side by side. Each cell shows the distance between two NMR
models. Darker / redder colours mean the models are more different. The diagonal
is zero (a model compared with itself).

In [ ]:
short = [lbl.replace("model_", "M") for lbl in mat_cheb.index]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

for ax, mat, title in [
    (ax1, mat_cheb, "Chebyshev (L∞)"),
    (ax2, mat_rms, "RMS"),
]:
    im = ax.imshow(mat.values, cmap="YlOrRd", aspect="equal")
    ax.set_xticks(range(len(mat)))
    ax.set_yticks(range(len(mat)))
    ax.set_xticklabels(short, rotation=45, ha="right", fontsize=7)
    ax.set_yticklabels(short, fontsize=7)
    ax.set_title(title, fontsize=12)
    fig.colorbar(im, ax=ax, shrink=0.7, label="BRI distance (Å)")

fig.suptitle(f"Pairwise BRI Distances — NMR Ensemble ({nmr_entry.pdb_id})",
             fontsize=13, y=1.02)
fig.tight_layout()
plt.show()

# Most and least similar model pairs (Chebyshev)
d = dist_cheb.sort_values("distance")
print(f"Most similar:   {d.iloc[0]['pdb_id1']} ↔ {d.iloc[0]['pdb_id2']}  "
      f"(Chebyshev = {d.iloc[0]['distance']:.3f} Å)")
print(f"Most different: {d.iloc[-1]['pdb_id1']} ↔ {d.iloc[-1]['pdb_id2']}  "
      f"(Chebyshev = {d.iloc[-1]['distance']:.3f} Å)")

### Question: Compare the matrices, what conclusions can you make?

## 5. Projections approach: Mean-Variance Scatter Projection

When comparing multiple distinct protein chains, or an ensemble of multiple models, what methods can you suggest to reduce the complexity of the structure?

One of the ways is to consider the first two moments, the **mean** and the **standard deviation** of the invariants.  
Function `plot_invariant_scatter()` collapses each 9-dimensional structure into a single point.

### 5.1 BRI projection

In [ ]:
from bri.workflow import plot_invariant_scatter

# Each point is one NMR model.
# Tight cluster  → invariant is consistent across models (rigid backbone).
# Spread-out     → invariant varies between models (flexible backbone).
fig = plot_invariant_scatter(nmr_bri, BRI_COLUMNS)
fig.suptitle("BRI Statistics per Model — 2K4P NMR Ensemble",
             fontsize=13, y=1.02)
plt.show()

### Question: what can you colnclude from these plots?

### Question: finish the following sentences
1. **Tight cluster of points** means that   
2. **Spread-out points** mean that
3. The bonuses of this approach:
4. The limitations of this approach:

### 5.2 LAI projection

The **Length-Angle Invariant** (LAI) translates BRI coordinates into chemically
meaningful quantities: bond lengths, bond angles, and torsion angles. Plotting
their statistics gives an more interpretable view of backbone flexibility.

### Question: finish the following sentences

- `tau(NA)`, `tau(AC)`, `tau(CN)` — torsion angles (related to φ, ψ, ω).
  Large spread means that   
- `length(N)`, `length(A)`, `length(C)` — bond lengths. Large spread means that  

In [ ]:
# Compute LAI for every model
from bri.invariant import LAI_COLUMNS
lai_frames = []
for chain in poly_chains:
    inv = chain.get_invariant(invariant_type="lai")
    inv["pdb_id"] = f"model_{chain.model_id}"
    lai_frames.append(inv)

nmr_lai = pd.concat(lai_frames, ignore_index=True)

print(f"LAI columns: {LAI_COLUMNS}")
print(f"LAI table: {len(nmr_lai)} rows")

In [ ]:
# Same function — just pass LAI columns and a different colour
fig = plot_invariant_scatter(nmr_lai, LAI_COLUMNS,
                             color="darkgreen", edge_color="forestgreen")
fig.suptitle("LAI Statistics per Model — 2K4P NMR Ensemble\n",
             fontsize=13, y=1.02)
plt.show()

### Question: what can you conclude form these plots?


## 6. Batch workflow — processing a folder of PDB files

For many `.pdb` files, the `bri.workflow` module is designed to handle everything, including **parallel processing** across CPU cores:

| Function | Input | Output |
|----------|-------|--------|
| `compute_dir_invariants(input_dir, output_dir)` | Folder of `.pdb` files | One `*_inv.csv` per file (BRI + LAI) |
| `compute_distance_matrix(data, output_dir)` | DataFrame or folder of CSVs | Distance matrices (per chain length) |
| `scatter_projection(data, output_dir)` | DataFrame or folder of CSVs | Statistical scatter plots (BRI + LAI) |

Distance matrices are produced **per chain length** because BRI comparison
requires chains of the same length.

> **Tip:** `compute_distance_matrix()` and `scatter_projection()` also accept
> a DataFrame (or list of DataFrames) directly — no need to save CSVs first.
> Omit `output_dir` to get results returned in memory instead of written to disk.

In [ ]:
import tempfile
from pathlib import Path
import biotite.database.rcsb as rcsb
from bri.workflow import (
    compute_dir_invariants,
    compute_distance_matrix,
    scatter_projection,
)

tmp = Path('tutorial_output')
pdb_dir = tmp / "pdbs"
pdb_dir.mkdir()

# Download a structure from RCSB for the demo.
# In practice, just point pdb_dir at your own folder of .pdb files.
rcsb.fetch("2k4p", "pdb", target_path=str(pdb_dir))

inv_dir = tmp / "inv"
dist_dir = tmp / "dist"
proj_dir = tmp / "proj"

# Step 1 — compute invariants for all .pdb files
n = compute_dir_invariants(pdb_dir, inv_dir)
print(f"Step 1 — invariants:  {n} file(s) → "
      f"{sorted(f.name for f in inv_dir.glob('*.csv'))}")

# Step 2 — distance matrices (grouped by chain length)
compute_distance_matrix(inv_dir, dist_dir)
print(f"Step 2 — matrices:    "
      f"{sorted(f.name for f in dist_dir.glob('*.csv'))}")

# Step 3 — statistical projections (BRI + LAI)
scatter_projection(inv_dir, proj_dir)
print(f"Step 3 — projections: "
      f"{sorted(f.name for f in proj_dir.iterdir())}")

print("\n✓ Batch workflow complete (files were written to a temp folder).")

In [ ]:
# The same functions also work with DataFrames directly — no CSV files needed:
from bri.workflow import compute_distance_matrix

matrices = compute_distance_matrix(nmr_bri)  # returns dict of {filename: matrix}
print(f"From in-memory DataFrame: {len(matrices)} matrices")
for name, mat in matrices.items():
    short_name = name.replace("distance_matrix_BRI_", "").replace(".csv", "")
    print(f"  {short_name}: {mat.shape[0]} × {mat.shape[1]} "
          f"(symmetric: {np.allclose(mat.values, mat.values.T)})")

## 7. Summary

### What we did

1. **Prepared** two datasets — hemoglobin (different-length chains) and an NMR
   ensemble (same-length chains).
2. **Overlaid** BRI profiles to see *where* along the sequence two chains differ.
3. **Computed pairwise distances** with two metrics (Chebyshev vs RMS) and
   visualised them as heat-maps.
4. **Projected statistics** onto scatter plots — for both BRI (coordinates) and
   LAI (bond lengths, angles, torsions) — to identify rigid vs flexible regions.
5. **Ran the batch workflow** to process a directory of PDB files end-to-end.

### Quick reference — comparison functions

| You want to … | Function |
|---------------|----------|
| Compare all chains pairwise | `group_invariant_compare(data, metric="chebyshev")` |
| Use RMS instead of Chebyshev | `group_invariant_compare(data, metric="rms")` |
| Convert distances to a matrix | `distance_table_to_matrix(dist_table)` |
| Plot invariant scatter (one call) | `plot_invariant_scatter(data, cols)` |
| Compute all distance matrices | `compute_distance_matrix(data)` |
| Compute all stats + plots | `scatter_projection(data)` |
| Process a folder of PDBs | `compute_dir_invariants(input_dir, output_dir)` |

> All functions in the table accept a DataFrame, a list of DataFrames, or a
> directory path. Pass `output_dir` to save results to disk; omit it to get
> them returned in memory.

### Key takeaways

- **No alignment needed** — BRI distances are rotation-invariant.
- **Chebyshev** = worst-case mismatch; **RMS** = average mismatch.
- Distance matrices require **same-length** chains; scatter projections work
  for **any** chains.
- **Bond lengths vary least**, **torsion angles vary most** — exactly what
  chemistry predicts.
- The batch workflow (`bri.workflow`) automates everything for large datasets.

> **Next steps:** Try loading your own structures, or explore
> `bri.invariant_compare` for nearest-neighbour search and Lipschitz constants.